In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import os
import torch
import os
import time
import pickle
import matplotlib.pyplot as plt
from scipy.stats import norm

from e_1_run_cvae import train_chunk, alarm
from e_1_run_cvae_time_check import train_chunk_time_check
#from e_2_CVAE_norm import CVAE as CVAE_norm
#from e_1_run_cvae_norm import train_chunk_norm

# global var
S0 = 1.0
K = 1.0
r = 0.03
sigma = np.sqrt(0.05)
T = 1.5
BS_eta = (S0, K, r, sigma, T)
# S0, K, r, kappa, theta, xi, rho, Y0, T = Hes_eta
Hes_eta = (S0, K, r, 2, 0.05, 0.5, -0.7, 0.05, T)

B = 0.8 # down-and-out must B < S0 and B < K
model_type = 'bs' # bs, bs_clip, hes, hes_clip
# barr_type = 'van' # van or barr
# opt_type = 'call' # call or put
# chunk_dir = f"/mnt/d/bs_chunks_correction/" if model_type == 'bs' else f"/mnt/d/hes_chunks_correction/"
# eta_path = "/mnt/d/bs_eta_basic.h5" if model_type == 'bs' else "/mnt/d/hes_eta_basic.h5"

if not((B < S0) & (B < K)):
    raise ValueError("down-and-out : B should be smaller than S0 and K")

# if not(opt_type == 'call' or  opt_type == 'put'):
#     raise ValueError("option_type must be 'call' or 'put'")

# if not(barr_type == 'van' or  barr_type == 'barr'):
#     raise ValueError("barr_type must be 'van' or 'barr'")

if not(model_type == 'hes' or  model_type == 'bs' or model_type == 'bs_clip' or model_type == 'hes_clip'):
    raise ValueError("model_type must be 'hes' or 'bs'")

if not(torch.cuda.is_available()):
    raise ValueError("CUDA is not available")
device = torch.device("cuda")

bs_stats = {
    "x_mean": -0.1045446063,
    "x_std": 0.6455563393,
    "m_mean": -0.4579059199,
    "m_std": 0.5553496410
}

if model_type == 'hes':
    test_etas = [0.03, 2.0,  0.05, 0.5, -0.7, 0.05, 1.5]
    eta_keys  = ['r', 'lambda', 'v_bar', 'xi', 'rho', 'Y0', 'T']
else: # model_type = 'bs'
    test_etas = [r, sigma, T]
    eta_keys  = ['r', 'sigma', 'T']

# if model_type == 'hes':
#     if barr_type == 'barr':
#         if opt_type == 'call':
#             bench_price = 0.115733
#         else: # put
#             bench_price = 0.005170
#     else: # van
#         if opt_type == 'call':
#             bench_price = 0.124491
#         else: # put
#             bench_price = 0.080488

# elif model_type == 'bs':
#     if barr_type == 'barr':
#         if opt_type == 'call':
#             bench_price = 0.123493
#         else: # put
#             bench_price = 0.009535
#     else: # van
#         if opt_type == 'call':
#             bench_price = 0.129944
#         else: # put
#             bench_price = 0.085942

/home/enjongoopee/.local/lib/python3.12/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


# training

In [18]:
# CVAE training settings
dim_z       = 2 # 2, 4, 8
hidden_dims = [1024, 512, 256] # [128, 128, 64], [512, 256, 128], [1024, 512, 256], [1024. 512, 256, 128], [2048, 1024, 512]
batch_size  = 8192 # 1024, 2048, 128-4096, 8192, 16384, 32768, 65536
bn_chunks   = None # None or num
use_bn      = True if bn_chunks is not None else False
lr          = 1e-5 # 수렴속도 1e-4 < 1e-5
l1          = 16
lr2         = 4e-6
l2          = 17
lr3         = 1e-6
l3          = 4
beta        = 1
warmup_chunks = None # None or num
num_chunks  = 94 # 1 chunk train : 2m
validation_chunk_idxs = [15,24,78]
val_every_chunks = 97
memory_on_gpu = True
cvae_type = "base" # "base" or "barr_weight"
weight_mode = "barrier_put" # "barrier_put" or "barrier_near"
weight_alpha = 3.0
weight_h = 0.01 # 0.01, 0.03, 0.04
weight_normalize = True
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1552.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{validation_chunk_idxs}_chunk1646.pt"

if 'base' in save_path:
    raise ValueError("base는 save_path에 포함되면 안됨.")

real_lr = lr
if str(lr2) in save_path:
    real_lr = lr2
elif str(lr3) in save_path:
    real_lr = lr3

In [ ]:
time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K, 
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_1024_8192_None_1e-05_1_[15, 24, 78]_chunk1552.pt | 완료 chunks=1552
learning rate : 4e-06
학습 시작 | 이번 실행 chunks=94 | 진행 chunks=1552->1646 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=97 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step  1553 | epoch   17 chunk   1/97 | file_idx  48 | BN off    | beta_eff: 1.0000 | Recon: -5.6622 | KL: 4.9059 | Total: -0.7563
Chunk step  1554 | epoch   17 chunk   2/97 | file_idx  52 | BN off    | beta_eff: 1.0000 | Recon: -5.6939 | KL: 4.9450 | Total: -0.7489
Chunk step  1555 | epoch   17 chunk   3/97 | file_idx  53 | BN off    | beta_eff: 1.0000 | Recon: -5.7073 | KL: 4.9544 | Total: -0.7530
Chunk step  1556 | epoch   17 chunk   4/97 | file_idx  50 | BN off    | beta_eff: 1.0000 | Recon: -5.7191 | KL: 4.9628 | Total: -0.7563
Chunk step  1557 | epoch   17 chunk   5/97 | file_idx  29 | BN off    | beta_eff: 1.0000 | Recon:

In [ ]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{validation_chunk_idxs}_chunk1646.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{validation_chunk_idxs}_chunk1649.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

alarm()

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_1024_8192_None_1e-05_1_[15, 24, 78]_chunk1355.pt | 완료 chunks=1355
learning rate : 1e-05
학습 시작 | 이번 실행 chunks=3 | 진행 chunks=1355->1358 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step  1356 | epoch   14 chunk  95/97 | file_idx  67 | BN off    | beta_eff: 1.0000 | Recon: -5.5895 | KL: 4.8409 | Total: -0.7486
Validation @ chunk  1356 | Recon: -5.5865 | KL: 4.8443 | Total: -0.7421 | KL_dim: [1.496663, 3.347634]
Chunk step  1357 | epoch   14 chunk  96/97 | file_idx  36 | BN off    | beta_eff: 1.0000 | Recon: -5.5906 | KL: 4.8381 | Total: -0.7524
Validation @ chunk  1357 | Recon: -5.5893 | KL: 4.8427 | Total: -0.7466 | KL_dim: [1.49578, 3.346874]
Chunk step  1358 | epoch   14 chunk  97/97 | file_idx  63 | BN off    | beta_eff: 1.0000 | Recon: -5.5913 | KL: 4.8394 | Total: -0.7520
Validation @ chunk  1358 | Rec

In [19]:
# CVAE training settings
dim_z       = 2 # 2, 4, 8
hidden_dims = [1024, 512, 256] # [128, 128, 64], [512, 256, 128], [1024, 512, 256], [1024. 512, 256, 128], [2048, 1024, 512]
batch_size  = 32768 # 1024, 2048, 128-4096, 8192, 16384, 32768, 65536
bn_chunks   = None # None or num
use_bn      = True if bn_chunks is not None else False
lr          = 1e-5 # 수렴속도 1e-4 < 1e-5
l1          = 5
lr2         = 1e-5
l2          = 6
lr3         = 1e-6
l3          = 4
beta        = 1
warmup_chunks = None # None or num
num_chunks  = 94 # 1 chunk train : 2m
validation_chunk_idxs = [15,24,78]
val_every_chunks = 97
memory_on_gpu = True
cvae_type = "base" # "base" or "barr_weight"
weight_mode = "barrier_put" # "barrier_put" or "barrier_near"
weight_alpha = 3.0
weight_h = 0.01 # 0.01, 0.03, 0.04
weight_normalize = True
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1552.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1646.pt"

if 'base' in save_path:
    raise ValueError("base는 save_path에 포함되면 안됨.")

In [ ]:
time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K, 
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_1024_32768_None_1e-05_1_[15, 24, 78]_chunk1552.pt | 완료 chunks=1552
learning rate : 1e-05
학습 시작 | 이번 실행 chunks=94 | 진행 chunks=1552->1646 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=97 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step  1553 | epoch   17 chunk   1/97 | file_idx  48 | BN off    | beta_eff: 1.0000 | Recon: -5.6027 | KL: 4.8506 | Total: -0.7521
Chunk step  1554 | epoch   17 chunk   2/97 | file_idx  52 | BN off    | beta_eff: 1.0000 | Recon: -5.5936 | KL: 4.8498 | Total: -0.7438
Chunk step  1555 | epoch   17 chunk   3/97 | file_idx  53 | BN off    | beta_eff: 1.0000 | Recon: -5.5898 | KL: 4.8423 | Total: -0.7475
Chunk step  1556 | epoch   17 chunk   4/97 | file_idx  50 | BN off    | beta_eff: 1.0000 | Recon: -5.5980 | KL: 4.8467 | Total: -0.7513
Chunk step  1557 | epoch   17 chunk   5/97 | file_idx  29 | BN off    | beta_eff: 1.0000 | Recon

In [ ]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1646.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1649.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

alarm()

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_1024_32768_None_1e-05_1_[15, 24, 78]_chunk1646.pt | 완료 chunks=1646
learning rate : 1e-05
학습 시작 | 이번 실행 chunks=3 | 진행 chunks=1646->1649 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step  1647 | epoch   17 chunk  95/97 | file_idx  85 | BN off    | beta_eff: 1.0000 | Recon: -5.6046 | KL: 4.8692 | Total: -0.7354
Validation @ chunk  1647 | Recon: -5.6123 | KL: 4.8654 | Total: -0.7470 | KL_dim: [1.503357, 3.361987]
Chunk step  1648 | epoch   17 chunk  96/97 | file_idx  77 | BN off    | beta_eff: 1.0000 | Recon: -5.6121 | KL: 4.8546 | Total: -0.7574
Validation @ chunk  1648 | Recon: -5.6091 | KL: 4.8652 | Total: -0.7439 | KL_dim: [1.504198, 3.361013]
Chunk step  1649 | epoch   17 chunk  97/97 | file_idx  88 | BN off    | beta_eff: 1.0000 | Recon: -5.6055 | KL: 4.8677 | Total: -0.7378
Validation @ chunk  1649 | R

In [ ]:
num_chunks  = 94
val_every_chunks = 97
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1649.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1743.pt"

time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K, 
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_1024_32768_None_1e-05_1_[15, 24, 78]_chunk1649.pt | 완료 chunks=1649
learning rate : 1e-05
학습 시작 | 이번 실행 chunks=94 | 진행 chunks=1649->1743 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=97 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step  1650 | epoch   18 chunk   1/97 | file_idx  17 | BN off    | beta_eff: 1.0000 | Recon: -5.6037 | KL: 4.8571 | Total: -0.7466
Chunk step  1651 | epoch   18 chunk   2/97 | file_idx  76 | BN off    | beta_eff: 1.0000 | Recon: -5.5991 | KL: 4.8578 | Total: -0.7413
Chunk step  1652 | epoch   18 chunk   3/97 | file_idx  89 | BN off    | beta_eff: 1.0000 | Recon: -5.6067 | KL: 4.8618 | Total: -0.7449
Chunk step  1653 | epoch   18 chunk   4/97 | file_idx  50 | BN off    | beta_eff: 1.0000 | Recon: -5.6065 | KL: 4.8558 | Total: -0.7507
Chunk step  1654 | epoch   18 chunk   5/97 | file_idx  94 | BN off    | beta_eff: 1.0000 | Recon

In [ ]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1743.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1746.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

alarm()

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_1024_32768_None_1e-05_1_[15, 24, 78]_chunk1743.pt | 완료 chunks=1743
learning rate : 1e-05
학습 시작 | 이번 실행 chunks=3 | 진행 chunks=1743->1746 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step  1744 | epoch   18 chunk  95/97 | file_idx   9 | BN off    | beta_eff: 1.0000 | Recon: -5.6174 | KL: 4.8538 | Total: -0.7636
Validation @ chunk  1744 | Recon: -5.6137 | KL: 4.8684 | Total: -0.7453 | KL_dim: [1.497586, 3.370832]
Chunk step  1745 | epoch   18 chunk  96/97 | file_idx  37 | BN off    | beta_eff: 1.0000 | Recon: -5.6089 | KL: 4.8674 | Total: -0.7414
Validation @ chunk  1745 | Recon: -5.6163 | KL: 4.8697 | Total: -0.7465 | KL_dim: [1.502183, 3.367547]
Chunk step  1746 | epoch   18 chunk  97/97 | file_idx  86 | BN off    | beta_eff: 1.0000 | Recon: -5.6182 | KL: 4.8621 | Total: -0.7561
Validation @ chunk  1746 | R

In [26]:
# CVAE training settings
dim_z       = 4 # 2, 4, 8
hidden_dims = [1024, 512, 256] # [128, 128, 64], [512, 256, 128], [1024, 512, 256], [1024. 512, 256, 128], [2048, 1024, 512]
batch_size  = 32768 # 1024, 2048, 128-4096, 8192, 16384, 32768, 65536
bn_chunks   = None # None or num
use_bn      = True if bn_chunks is not None else False
lr          = 1e-5 # 수렴속도 1e-4 < 1e-5
l1          = 5
lr2         = 1e-5
l2          = 6
lr3         = 1e-6
l3          = 4
beta        = 1
warmup_chunks = None # None or num
num_chunks  = 94 # 1 chunk train : 2m
validation_chunk_idxs = [15,24,78]
val_every_chunks = 97
memory_on_gpu = True
cvae_type = "base" # "base" or "barr_weight"
weight_mode = "barrier_put" # "barrier_put" or "barrier_near"
weight_alpha = 3.0
weight_h = 0.01 # 0.01, 0.03, 0.04
weight_normalize = True
resume_path = None # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk94.pt"

if 'base' in save_path:
    raise ValueError("base는 save_path에 포함되면 안됨.")

In [ ]:
time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K, 
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

learning rate : 1e-05
학습 시작 | 이번 실행 chunks=94 | 진행 chunks=0->94 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=97 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step     1 | epoch    1 chunk   1/97 | file_idx  88 | BN off    | beta_eff: 1.0000 | Recon: -1.2891 | KL: 1.4264 | Total: 0.1373
Chunk step     2 | epoch    1 chunk   2/97 | file_idx   8 | BN off    | beta_eff: 1.0000 | Recon: -2.7581 | KL: 2.4325 | Total: -0.3256
Chunk step     3 | epoch    1 chunk   3/97 | file_idx  75 | BN off    | beta_eff: 1.0000 | Recon: -3.2480 | KL: 2.8385 | Total: -0.4096
Chunk step     4 | epoch    1 chunk   4/97 | file_idx  41 | BN off    | beta_eff: 1.0000 | Recon: -3.4829 | KL: 3.0402 | Total: -0.4426
Chunk step     5 | epoch    1 chunk   5/97 | file_idx  17 | BN off    | beta_eff: 1.0000 | Recon: -3.6623 | KL: 3.1954 | Total: -0.4669
Chunk step     6 | epoch    1 chunk   6/97 | file_idx  66 | BN off  

In [ ]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk94.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk97.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

alarm()

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_4_1024_32768_None_1e-05_1_[15, 24, 78]_chunk94.pt | 완료 chunks=94
learning rate : 1e-05
학습 시작 | 이번 실행 chunks=3 | 진행 chunks=94->97 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step    95 | epoch    1 chunk  95/97 | file_idx  85 | BN off    | beta_eff: 1.0000 | Recon: -4.5749 | KL: 3.8941 | Total: -0.6807
Validation @ chunk    95 | Recon: -4.5762 | KL: 3.8966 | Total: -0.6796 | KL_dim: [2.629714, 0.000109, 1.266662, 0.000137]
Chunk step    96 | epoch    1 chunk  96/97 | file_idx  80 | BN off    | beta_eff: 1.0000 | Recon: -4.5818 | KL: 3.9025 | Total: -0.6793
Validation @ chunk    96 | Recon: -4.6395 | KL: 3.9379 | Total: -0.7016 | KL_dim: [2.651568, 0.00013, 1.286049, 0.000137]
Chunk step    97 | epoch    1 chunk  97/97 | file_idx  61 | BN off    | beta_eff: 1.0000 | Recon: -4.6212 | KL: 3.9269 | Total: -0.69

In [ ]:
num_chunks  = 94
val_every_chunks = 97
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk97.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk191.pt"

time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K, 
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_4_1024_32768_None_1e-05_1_[15, 24, 78]_chunk97.pt | 완료 chunks=97
learning rate : 1e-05
학습 시작 | 이번 실행 chunks=94 | 진행 chunks=97->191 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=97 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step    98 | epoch    2 chunk   1/97 | file_idx   7 | BN off    | beta_eff: 1.0000 | Recon: -4.6036 | KL: 3.9271 | Total: -0.6764
Chunk step    99 | epoch    2 chunk   2/97 | file_idx  14 | BN off    | beta_eff: 1.0000 | Recon: -4.5997 | KL: 3.9127 | Total: -0.6871
Chunk step   100 | epoch    2 chunk   3/97 | file_idx  47 | BN off    | beta_eff: 1.0000 | Recon: -4.6201 | KL: 3.9187 | Total: -0.7014
Chunk step   101 | epoch    2 chunk   4/97 | file_idx  79 | BN off    | beta_eff: 1.0000 | Recon: -4.6214 | KL: 3.9193 | Total: -0.7021
Chunk step   102 | epoch    2 chunk   5/97 | file_idx  83 | BN off    | beta_eff: 1.0000 | Recon: -4.62

In [ ]:
num_chunks  = 3
val_every_chunks = 1
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk191.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk194.pt"

time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K, 
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_4_1024_32768_None_1e-05_1_[15, 24, 78]_chunk191.pt | 완료 chunks=191
learning rate : 1e-05
학습 시작 | 이번 실행 chunks=3 | 진행 chunks=191->194 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step   192 | epoch    2 chunk  95/97 | file_idx  32 | BN off    | beta_eff: 1.0000 | Recon: -4.8546 | KL: 4.1325 | Total: -0.7222
Validation @ chunk   192 | Recon: -4.8606 | KL: 4.1783 | Total: -0.6823 | KL_dim: [2.752846, 0.003353, 1.421191, 0.000952]
Chunk step   193 | epoch    2 chunk  96/97 | file_idx  74 | BN off    | beta_eff: 1.0000 | Recon: -4.8658 | KL: 4.1372 | Total: -0.7285
Validation @ chunk   193 | Recon: -4.8530 | KL: 4.1524 | Total: -0.7006 | KL_dim: [2.751077, 0.000218, 1.400889, 0.000185]
Chunk step   194 | epoch    2 chunk  97/97 | file_idx  67 | BN off    | beta_eff: 1.0000 | Recon: -4.8379 | KL: 4.1236 | Total: 

In [ ]:
num_chunks  = 94
val_every_chunks = 97
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk194.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk288.pt"

time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K, 
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_4_1024_32768_None_1e-05_1_[15, 24, 78]_chunk194.pt | 완료 chunks=194
learning rate : 1e-05
학습 시작 | 이번 실행 chunks=94 | 진행 chunks=194->288 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=97 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step   195 | epoch    3 chunk   1/97 | file_idx  25 | BN off    | beta_eff: 1.0000 | Recon: -4.8578 | KL: 4.1419 | Total: -0.7159
Chunk step   196 | epoch    3 chunk   2/97 | file_idx  80 | BN off    | beta_eff: 1.0000 | Recon: -4.8715 | KL: 4.1660 | Total: -0.7055
Chunk step   197 | epoch    3 chunk   3/97 | file_idx  84 | BN off    | beta_eff: 1.0000 | Recon: -4.8538 | KL: 4.1552 | Total: -0.6986
Chunk step   198 | epoch    3 chunk   4/97 | file_idx   1 | BN off    | beta_eff: 1.0000 | Recon: -4.8710 | KL: 4.1570 | Total: -0.7140
Chunk step   199 | epoch    3 chunk   5/97 | file_idx  60 | BN off    | beta_eff: 1.0000 | Recon: -4

KeyboardInterrupt: 

In [ ]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk288.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk291.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_1024_8192_None_1e-05_1_[15, 24, 78]_chunk1161.pt | 완료 chunks=1161
learning rate : 1e-05
학습 시작 | 이번 실행 chunks=3 | 진행 chunks=1161->1164 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step  1162 | epoch   12 chunk  95/97 | file_idx  42 | BN off    | beta_eff: 1.0000 | Recon: -5.5729 | KL: 4.8353 | Total: -0.7375
Validation @ chunk  1162 | Recon: -5.5794 | KL: 4.8330 | Total: -0.7464 | KL_dim: [1.495629, 3.337391]
Chunk step  1163 | epoch   12 chunk  96/97 | file_idx  30 | BN off    | beta_eff: 1.0000 | Recon: -5.5713 | KL: 4.8401 | Total: -0.7312
Validation @ chunk  1163 | Recon: -5.5735 | KL: 4.8285 | Total: -0.7450 | KL_dim: [1.493433, 3.335078]
Chunk step  1164 | epoch   12 chunk  97/97 | file_idx  58 | BN off    | beta_eff: 1.0000 | Recon: -5.5766 | KL: 4.8325 | Total: -0.7441
Validation @ chunk  1164 | Re

In [ ]:
num_chunks  = 94
val_every_chunks = 97
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk291.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk385.pt"

time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K, 
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_1024_8192_None_1e-05_1_[15, 24, 78]_chunk1164.pt | 완료 chunks=1164
learning rate : 1e-05
학습 시작 | 이번 실행 chunks=94 | 진행 chunks=1164->1258 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=97 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step  1165 | epoch   13 chunk   1/97 | file_idx  46 | BN off    | beta_eff: 1.0000 | Recon: -5.5757 | KL: 4.8324 | Total: -0.7433
Chunk step  1166 | epoch   13 chunk   2/97 | file_idx  18 | BN off    | beta_eff: 1.0000 | Recon: -5.5775 | KL: 4.8274 | Total: -0.7500
Chunk step  1167 | epoch   13 chunk   3/97 | file_idx  27 | BN off    | beta_eff: 1.0000 | Recon: -5.5755 | KL: 4.8237 | Total: -0.7518
Chunk step  1168 | epoch   13 chunk   4/97 | file_idx   1 | BN off    | beta_eff: 1.0000 | Recon: -5.5742 | KL: 4.8322 | Total: -0.7420
Chunk step  1169 | epoch   13 chunk   5/97 | file_idx  57 | BN off    | beta_eff: 1.0000 | Recon:

In [ ]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk385.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk388.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_1024_8192_None_1e-05_1_[15, 24, 78]_chunk1258.pt | 완료 chunks=1258
learning rate : 1e-05
학습 시작 | 이번 실행 chunks=3 | 진행 chunks=1258->1261 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step  1259 | epoch   13 chunk  95/97 | file_idx  59 | BN off    | beta_eff: 1.0000 | Recon: -5.5862 | KL: 4.8382 | Total: -0.7479
Validation @ chunk  1259 | Recon: -5.5859 | KL: 4.8412 | Total: -0.7447 | KL_dim: [1.495614, 3.345606]
Chunk step  1260 | epoch   13 chunk  96/97 | file_idx   9 | BN off    | beta_eff: 1.0000 | Recon: -5.5849 | KL: 4.8205 | Total: -0.7643
Validation @ chunk  1260 | Recon: -5.5911 | KL: 4.8477 | Total: -0.7434 | KL_dim: [1.497528, 3.350124]
Chunk step  1261 | epoch   13 chunk  97/97 | file_idx  62 | BN off    | beta_eff: 1.0000 | Recon: -5.5801 | KL: 4.8422 | Total: -0.7379
Validation @ chunk  1261 | Re

In [ ]:
num_chunks  = 94
val_every_chunks = 97
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk388.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk482.pt"

time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K, 
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_1024_32768_None_1e-05_1_[15, 24, 78]_chunk1261.pt | 완료 chunks=1261
learning rate : 1e-05
학습 시작 | 이번 실행 chunks=94 | 진행 chunks=1261->1355 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=97 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step  1262 | epoch   14 chunk   1/97 | file_idx  92 | BN off    | beta_eff: 1.0000 | Recon: -5.5444 | KL: 4.7983 | Total: -0.7461
Chunk step  1263 | epoch   14 chunk   2/97 | file_idx  81 | BN off    | beta_eff: 1.0000 | Recon: -5.5287 | KL: 4.7986 | Total: -0.7301
Chunk step  1264 | epoch   14 chunk   3/97 | file_idx  83 | BN off    | beta_eff: 1.0000 | Recon: -5.5321 | KL: 4.7959 | Total: -0.7362
Chunk step  1265 | epoch   14 chunk   4/97 | file_idx  60 | BN off    | beta_eff: 1.0000 | Recon: -5.5404 | KL: 4.7978 | Total: -0.7425
Chunk step  1266 | epoch   14 chunk   5/97 | file_idx   2 | BN off    | beta_eff: 1.0000 | Recon

In [ ]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk482.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk485.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_1024_32768_None_1e-05_1_[15, 24, 78]_chunk1355.pt | 완료 chunks=1355
learning rate : 1e-05
학습 시작 | 이번 실행 chunks=3 | 진행 chunks=1355->1358 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step  1356 | epoch   14 chunk  95/97 | file_idx  67 | BN off    | beta_eff: 1.0000 | Recon: -5.5579 | KL: 4.8113 | Total: -0.7466
Validation @ chunk  1356 | Recon: -5.5528 | KL: 4.8105 | Total: -0.7423 | KL_dim: [1.511684, 3.298795]
Chunk step  1357 | epoch   14 chunk  96/97 | file_idx  36 | BN off    | beta_eff: 1.0000 | Recon: -5.5629 | KL: 4.8126 | Total: -0.7504
Validation @ chunk  1357 | Recon: -5.5652 | KL: 4.8186 | Total: -0.7467 | KL_dim: [1.516792, 3.301764]
Chunk step  1358 | epoch   14 chunk  97/97 | file_idx  63 | BN off    | beta_eff: 1.0000 | Recon: -5.5666 | KL: 4.8164 | Total: -0.7502
Validation @ chunk  1358 | R

In [ ]:
num_chunks  = 94
val_every_chunks = 97
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk582.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk676.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_1024_32768_None_1e-05_1_[15, 24, 78]_chunk582.pt | 완료 chunks=582
learning rate : 1e-05
학습 시작 | 이번 실행 chunks=94 | 진행 chunks=582->676 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=97 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step   583 | epoch    7 chunk   1/97 | file_idx  89 | BN off    | beta_eff: 1.0000 | Recon: -5.1182 | KL: 4.3878 | Total: -0.7304
Chunk step   584 | epoch    7 chunk   2/97 | file_idx  50 | BN off    | beta_eff: 1.0000 | Recon: -5.1230 | KL: 4.3862 | Total: -0.7368
Chunk step   585 | epoch    7 chunk   3/97 | file_idx  28 | BN off    | beta_eff: 1.0000 | Recon: -5.1284 | KL: 4.3941 | Total: -0.7344
Chunk step   586 | epoch    7 chunk   4/97 | file_idx  59 | BN off    | beta_eff: 1.0000 | Recon: -5.1274 | KL: 4.3943 | Total: -0.7331
Chunk step   587 | epoch    7 chunk   5/97 | file_idx  55 | BN off    | beta_eff: 1.0000 | Recon: -5

In [ ]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk676.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk679.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_1024_32768_None_1e-05_1_[15, 24, 78]_chunk676.pt | 완료 chunks=676
learning rate : 1e-05
학습 시작 | 이번 실행 chunks=3 | 진행 chunks=676->679 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step   677 | epoch    7 chunk  95/97 | file_idx  22 | BN off    | beta_eff: 1.0000 | Recon: -5.1510 | KL: 4.4168 | Total: -0.7342
Validation @ chunk   677 | Recon: -5.1436 | KL: 4.4231 | Total: -0.7205 | KL_dim: [1.548143, 2.874961]
Chunk step   678 | epoch    7 chunk  96/97 | file_idx  43 | BN off    | beta_eff: 1.0000 | Recon: -5.1604 | KL: 4.4166 | Total: -0.7438
Validation @ chunk   678 | Recon: -5.1616 | KL: 4.4383 | Total: -0.7232 | KL_dim: [1.557193, 2.881129]
Chunk step   679 | epoch    7 chunk  97/97 | file_idx  11 | BN off    | beta_eff: 1.0000 | Recon: -5.1600 | KL: 4.4141 | Total: -0.7459
Validation @ chunk   679 | Recon

In [ ]:
num_chunks  = 94
val_every_chunks = 97
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk679.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk773.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_1024_32768_None_1e-05_1_[15, 24, 78]_chunk679.pt | 완료 chunks=679
learning rate : 1e-05
학습 시작 | 이번 실행 chunks=94 | 진행 chunks=679->773 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=97 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step   680 | epoch    8 chunk   1/97 | file_idx  16 | BN off    | beta_eff: 1.0000 | Recon: -5.1553 | KL: 4.4295 | Total: -0.7258
Chunk step   681 | epoch    8 chunk   2/97 | file_idx  77 | BN off    | beta_eff: 1.0000 | Recon: -5.1500 | KL: 4.4064 | Total: -0.7436
Chunk step   682 | epoch    8 chunk   3/97 | file_idx  66 | BN off    | beta_eff: 1.0000 | Recon: -5.1468 | KL: 4.3894 | Total: -0.7574
Chunk step   683 | epoch    8 chunk   4/97 | file_idx  38 | BN off    | beta_eff: 1.0000 | Recon: -5.1489 | KL: 4.4311 | Total: -0.7178
Chunk step   684 | epoch    8 chunk   5/97 | file_idx  17 | BN off    | beta_eff: 1.0000 | Recon: -5

In [ ]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk773.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk776.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_1024_32768_None_1e-05_1_[15, 24, 78]_chunk773.pt | 완료 chunks=773
learning rate : 1e-05
학습 시작 | 이번 실행 chunks=3 | 진행 chunks=773->776 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step   774 | epoch    8 chunk  95/97 | file_idx  21 | BN off    | beta_eff: 1.0000 | Recon: -5.1766 | KL: 4.4516 | Total: -0.7250
Validation @ chunk   774 | Recon: -5.1859 | KL: 4.4548 | Total: -0.7311 | KL_dim: [1.561545, 2.893239]
Chunk step   775 | epoch    8 chunk  96/97 | file_idx  93 | BN off    | beta_eff: 1.0000 | Recon: -5.1806 | KL: 4.4392 | Total: -0.7413
Validation @ chunk   775 | Recon: -5.1891 | KL: 4.4569 | Total: -0.7322 | KL_dim: [1.562046, 2.894847]
Chunk step   776 | epoch    8 chunk  97/97 | file_idx   2 | BN off    | beta_eff: 1.0000 | Recon: -5.1838 | KL: 4.4542 | Total: -0.7296
Validation @ chunk   776 | Recon

In [ ]:
num_chunks  = 94
val_every_chunks = 97
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk776.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk870.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_1024_32768_None_1e-05_1_[15, 24, 78]_chunk776.pt | 완료 chunks=776
learning rate : 1e-05
학습 시작 | 이번 실행 chunks=94 | 진행 chunks=776->870 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=97 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step   777 | epoch    9 chunk   1/97 | file_idx  22 | BN off    | beta_eff: 1.0000 | Recon: -5.1804 | KL: 4.4444 | Total: -0.7360
Chunk step   778 | epoch    9 chunk   2/97 | file_idx  84 | BN off    | beta_eff: 1.0000 | Recon: -5.1824 | KL: 4.4645 | Total: -0.7179
Chunk step   779 | epoch    9 chunk   3/97 | file_idx  49 | BN off    | beta_eff: 1.0000 | Recon: -5.1861 | KL: 4.4598 | Total: -0.7263
Chunk step   780 | epoch    9 chunk   4/97 | file_idx  79 | BN off    | beta_eff: 1.0000 | Recon: -5.1870 | KL: 4.4451 | Total: -0.7419
Chunk step   781 | epoch    9 chunk   5/97 | file_idx  29 | BN off    | beta_eff: 1.0000 | Recon: -5

In [ ]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk870.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk873.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_1024_32768_None_1e-05_1_[15, 24, 78]_chunk870.pt | 완료 chunks=870
learning rate : 1e-05
학습 시작 | 이번 실행 chunks=3 | 진행 chunks=870->873 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step   871 | epoch    9 chunk  95/97 | file_idx  34 | BN off    | beta_eff: 1.0000 | Recon: -5.2241 | KL: 4.4862 | Total: -0.7379
Validation @ chunk   871 | Recon: -5.2187 | KL: 4.5011 | Total: -0.7176 | KL_dim: [1.574806, 2.926335]
Chunk step   872 | epoch    9 chunk  96/97 | file_idx  97 | BN off    | beta_eff: 1.0000 | Recon: -5.2171 | KL: 4.4948 | Total: -0.7224
Validation @ chunk   872 | Recon: -5.2275 | KL: 4.4915 | Total: -0.7360 | KL_dim: [1.572746, 2.918773]
Chunk step   873 | epoch    9 chunk  97/97 | file_idx  35 | BN off    | beta_eff: 1.0000 | Recon: -5.2249 | KL: 4.4696 | Total: -0.7553
Validation @ chunk   873 | Recon

In [ ]:
num_chunks  = 94
val_every_chunks = 97
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk873.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk967.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_1024_32768_None_1e-05_1_[15, 24, 78]_chunk873.pt | 완료 chunks=873
learning rate : 1e-05
학습 시작 | 이번 실행 chunks=94 | 진행 chunks=873->967 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=97 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step   874 | epoch   10 chunk   1/97 | file_idx  50 | BN off    | beta_eff: 1.0000 | Recon: -5.2237 | KL: 4.4824 | Total: -0.7413
Chunk step   875 | epoch   10 chunk   2/97 | file_idx  44 | BN off    | beta_eff: 1.0000 | Recon: -5.2293 | KL: 4.4898 | Total: -0.7395
Chunk step   876 | epoch   10 chunk   3/97 | file_idx  89 | BN off    | beta_eff: 1.0000 | Recon: -5.2296 | KL: 4.4948 | Total: -0.7348
Chunk step   877 | epoch   10 chunk   4/97 | file_idx  61 | BN off    | beta_eff: 1.0000 | Recon: -5.2231 | KL: 4.4876 | Total: -0.7355
Chunk step   878 | epoch   10 chunk   5/97 | file_idx  11 | BN off    | beta_eff: 1.0000 | Recon: -5

In [ ]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk967.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk970.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_1024_32768_None_1e-05_1_[15, 24, 78]_chunk967.pt | 완료 chunks=967
learning rate : 1e-05
학습 시작 | 이번 실행 chunks=3 | 진행 chunks=967->970 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step   968 | epoch   10 chunk  95/97 | file_idx  66 | BN off    | beta_eff: 1.0000 | Recon: -5.2995 | KL: 4.5369 | Total: -0.7625
Validation @ chunk   968 | Recon: -5.2982 | KL: 4.5592 | Total: -0.7390 | KL_dim: [1.579899, 2.979314]
Chunk step   969 | epoch   10 chunk  96/97 | file_idx  47 | BN off    | beta_eff: 1.0000 | Recon: -5.2997 | KL: 4.5548 | Total: -0.7450
Validation @ chunk   969 | Recon: -5.2950 | KL: 4.5602 | Total: -0.7348 | KL_dim: [1.581964, 2.97827]
Chunk step   970 | epoch   10 chunk  97/97 | file_idx  26 | BN off    | beta_eff: 1.0000 | Recon: -5.2994 | KL: 4.5459 | Total: -0.7534
Validation @ chunk   970 | Recon:

In [ ]:
num_chunks  = 94
val_every_chunks = 97
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk970.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1064.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_1024_32768_None_1e-05_1_[15, 24, 78]_chunk970.pt | 완료 chunks=970
learning rate : 1e-05
학습 시작 | 이번 실행 chunks=94 | 진행 chunks=970->1064 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=97 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step   971 | epoch   11 chunk   1/97 | file_idx  41 | BN off    | beta_eff: 1.0000 | Recon: -5.2931 | KL: 4.5574 | Total: -0.7358
Chunk step   972 | epoch   11 chunk   2/97 | file_idx  91 | BN off    | beta_eff: 1.0000 | Recon: -5.2955 | KL: 4.5433 | Total: -0.7523
Chunk step   973 | epoch   11 chunk   3/97 | file_idx  83 | BN off    | beta_eff: 1.0000 | Recon: -5.2919 | KL: 4.5621 | Total: -0.7298
Chunk step   974 | epoch   11 chunk   4/97 | file_idx  49 | BN off    | beta_eff: 1.0000 | Recon: -5.2943 | KL: 4.5650 | Total: -0.7292
Chunk step   975 | epoch   11 chunk   5/97 | file_idx  54 | BN off    | beta_eff: 1.0000 | Recon: -

In [ ]:
# CVAE training settings
num_chunks  = 3
val_every_chunks = 1
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1064.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1067.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")
alarm()

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/bs/cvae_bs_2_1024_32768_None_1e-05_1_[15, 24, 78]_chunk1064.pt | 완료 chunks=1064
learning rate : 1e-05
학습 시작 | 이번 실행 chunks=3 | 진행 chunks=1064->1067 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step  1065 | epoch   11 chunk  95/97 | file_idx  10 | BN off    | beta_eff: 1.0000 | Recon: -5.3881 | KL: 4.6381 | Total: -0.7500
Validation @ chunk  1065 | Recon: -5.3808 | KL: 4.6397 | Total: -0.7411 | KL_dim: [1.569698, 3.069986]
Chunk step  1066 | epoch   11 chunk  96/97 | file_idx  17 | BN off    | beta_eff: 1.0000 | Recon: -5.3854 | KL: 4.6441 | Total: -0.7413
Validation @ chunk  1066 | Recon: -5.3924 | KL: 4.6492 | Total: -0.7431 | KL_dim: [1.574079, 3.075169]
Chunk step  1067 | epoch   11 chunk  97/97 | file_idx  81 | BN off    | beta_eff: 1.0000 | Recon: -5.3851 | KL: 4.6586 | Total: -0.7265
Validation @ chunk  1067 | R

# time check

In [ ]:
# CVAE training settings
dim_z       = 2 # 8, 12
hidden_dims = [2048, 1024, 512] # [128, 128, 64], [256, 256, 128], [512, 256, 128], [1024, 512, 256], [2048, 1024, 512], [1024, 512, 256, 128]
batch_size  = 32768 # 1024, 2048, 4096, 8192, 16384, 32768, 65536
bn_chunks   = None # None or num
use_bn      = True if bn_chunks is not None else False
lr          = 1e-5 # 수렴속도 1e-4 < 1e-5
l1          = 5
lr2         = 1e-5
l2          = 6
lr3         = 1e-6
l3          = 4
beta        = 1
warmup_chunks = None # None or num
num_chunks  = 2 # 1 chunk train : 2m
validation_chunk_idxs = [15,24,78]
val_every_chunks = 97
memory_on_gpu = True
cvae_type = "base" # "base" or "barr_weight"
weight_mode = "barrier_put" # "barrier_put" or "barrier_near"
weight_alpha = 3.0
weight_h = 0.01 # 0.01, 0.03, 0.04
weight_normalize = True
resume_path = None # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk2.pt"

time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk_time_check(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

learning rate : 1e-05
GPU: NVIDIA GeForce RTX 4080 SUPER | cuda capability=(8, 9)
학습 시작 | 이번 실행 chunks=2 | 진행 chunks=0->2 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=97 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step     1 | epoch    1 chunk   1/97 | file_idx  88 | BN off    | beta_eff: 1.0000 | Recon: -1.2521 | KL: 1.3628 | Total: 0.1106
Time | total=127.42s | chunk_load=14.71s | load_wait=14.71s | gpu_move=0.12s | loader_init=0.00s | iter_init=0.00s | batch_fetch=0.00s (0.0000/batch) | h2d=0.00s (0.0000/batch) | fwd=45.48s (0.0445/batch) | bwd=80.01s (0.0782/batch) | clip=0.75s (0.0007/batch) | step=0.87s (0.0009/batch) | loss_item=0.29s | cleanup=0.00s | loop_overhead=0.00s (0.0000/batch) | batches=1023
Chunk step     2 | epoch    1 chunk   2/97 | file_idx   8 | BN off    | beta_eff: 1.0000 | Recon: -2.0999 | KL: 1.9110 | Total: -0.1890
Time | total=126.72s | chunk_load=15.17s | 